In [ ]:
from datetime import date, datetime
from models.binomial import AmericanCRRAnalyzer, EuropeanCRRAnalyzer

In [ ]:
def year_fraction(expiration_date: str, valuation_date: str | None = None, day_count: float = 365.0) -> float:
    expiration = datetime.strptime(expiration_date, "%Y-%m-%d").date()
    valuation = datetime.strptime(valuation_date, "%Y-%m-%d").date() if valuation_date else date.today()
    maturity = (expiration - valuation).days / day_count
    return max(maturity, 1e-8)


def prepare_contract(contract: dict) -> dict:
    normalized = dict(contract)
    if "time_to_maturity" not in normalized and "expiration_date" in normalized:
        normalized["time_to_maturity"] = year_fraction(
            expiration_date=normalized["expiration_date"],
            valuation_date=normalized.get("valuation_date"),
        )
    return normalized

In [38]:
contract = {
    "spot": 256.44,
    "strike": 260,
    "expiration_date": "2027-06-17",
    "risk_free_rate": 0.0415,
    "volatility": 0.30,
    "steps": 1000,
    "option_type": "call",
}

In [39]:
european_analyzer = EuropeanCRRAnalyzer(scheme="crr")
american_analyzer = AmericanCRRAnalyzer(scheme="crr")

In [40]:
normalized_contract = prepare_contract(contract)
result_european = european_analyzer.analyze(normalized_contract)
result_american = american_analyzer.analyze(normalized_contract)

In [41]:
print("European:", result_european)
print("American:", result_american)

European: {'price': 39.107422495895186, 'delta': 0.6131439094528464, 'gamma': 0.0022156314330654685, 'vega': 111.2667523328458, 'theta': -17.941306198375884, 'rho': 151.23393354855992}
American: {'price': 39.107422495895186, 'delta': 0.6131439094528464, 'gamma': 0.0022156314330654685, 'vega': 111.2667523328458, 'theta': -17.941306198375884, 'rho': 151.23393354855992}
